<a href="https://colab.research.google.com/github/suyaibalsifat/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suyaibalsifat/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!git clone https://github.com/SUYAIBALSIFAT/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 132 (delta 45), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 1.85 MiB | 15.51 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/flyrank-ml-internship


## 1. My lane as an ML task (type)

This is a ranking / scoring task. The output isn't one yes/no answer — it's a priority order across thousands of pages, so an editor with limited time knows which ones to look at first. That matches the "which ones first?" pattern from the ML task-type guide.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

Proxy target: needs_review = 1 if trend_direction is "down" AND impressions_90d >= 100 (a declining page that still has real search demand), else 0.
This is a proxy, not a true observed outcome — trend_direction is itself a bucket calculated from the current window, not a future result. A stronger version later (once I have the warehouse data) would use a FUTURE window: e.g. "declines further over the next 30 days" instead of "is currently declining." I'm using this proxy now because it's all the starter data supports, and I'll say so honestly in my write-up.

In [3]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["needs_review_proxy"] = ((df.trend_direction == "down") & (df.impressions_90d >= 100)).astype(int)

print("Positive rate (pages flagged):", round(df.needs_review_proxy.mean(), 4))
print(df["needs_review_proxy"].value_counts())

Positive rate (pages flagged): 0.4384
needs_review_proxy
0    16848
1    13152
Name: count, dtype: int64


## 3. Success metric

Precision@50 — of the top 50 pages my ranking puts first, how many are actually true positives (needs_review_proxy == 1)? I'm choosing this over plain accuracy because an editor only has capacity to review a fixed number of pages per week, so what matters is whether the TOP of the list is right, not whether every single page in the dataset is classified correctly.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

One row = one content page. Below are the columns I'll actually use as features/context, plus the proxy target column.

In [4]:
cols = ["content_id", "impressions_90d", "clicks_90d", "avg_position", "ctr",
        "trend_direction", "days_since_last_update", "word_count", "needs_review_proxy"]
df[cols].head(10)

,content_id,impressions_90d,clicks_90d,avg_position,ctr,trend_direction,days_since_last_update,word_count,needs_review_proxy
0,content_304f48230142,3803,29,10.6,0.76,down,20,3221.0,1
1,content_a1fb4e703a9e,15320,7,20.3,0.05,down,25,2481.0,1
2,content_9aa793d4d895,12581,11,36.5,0.09,down,20,3515.0,1
3,content_331d6c4de07b,11751,58,6.2,0.49,stable,22,NaN,0
4,content_d99b7a2d90ca,19140,24,44.0,0.13,down,14,2803.0,1
5,content_d4084a4bc775,3970,1,8.5,0.03,down,20,3080.0,1
6,content_9a34b442b552,20,0,7.0,0.00,down,20,3059.0,0
7,content_a63219c6e95a,1724,1,21.2,0.06,stable,22,NaN,0
8,content_5e6c160719bc,32574,29,46.0,0.09,down,20,3807.0,1
9,content_c27558df2b0c,1240,2,4.9,0.16,down,104,NaN,1


## 5. Why ML beats a fixed rule here

A fixed rule like "flag every page with trend_direction == down" would flag about 44% of all 30,000 pages — far more than any editor could review in a week, and it treats a page losing 2% traffic the same as one losing 60%. It also ignores other relevant signals like CTR, position, and freshness that interact with each other in ways that are hard to write as a clean if-statement. ML-01's own results back this up: the fixed baseline rule only hit ~24% precision@top-K, while a trained model reached ~74% — proof that combining multiple weak signals beats a single hand-written threshold here.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.